# Football contracts 500: dataset exploration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jackstev-edu/player-value-forecaster/blob/main/notebooks/01_dataset_eda.ipynb)

CMU 24-679 Project 1, Yunus Polatoglu and Jack Stevens. This notebook explores the dataset
behind the [Player Value Forecaster](https://huggingface.co/spaces/ypolatog/player-value-forecaster):
**500 contracts we collected by hand**, plus 500 augmented rows kept in a separate split.

Dataset card: https://huggingface.co/datasets/ypolatog/football-contracts-500. **Runtime → Run all** works on a free CPU
runtime in under a minute, with nothing to install.

In [ ]:
import os
import pandas as pd

# Reads the published Hugging Face dataset. PVF_LOCAL_DATASET points at a local copy instead
# (used to test the notebook before publishing).
DATASET = "ypolatog/football-contracts-500"
BASE = os.environ.get("PVF_LOCAL_DATASET", f"hf://datasets/{DATASET}")

def read(path):
    full = f"{BASE}/{path}"
    return pd.read_csv(full) if path.endswith(".csv") else pd.read_parquet(full)

manual = read("samples/manual.parquet")
augmented = read("samples/augmented.parquet")
contracts = read("contracts/manual.csv")
history = read("value_history.parquet")
print(f"manual {len(manual)} rows, augmented {len(augmented)} rows, "
      f"contracts {len(contracts)}, value history {len(history)} valuations")

## 1. Manual and augmented rows are kept apart

The rubric asks for augmented data to be separated. Every manual row is a distinct player
(500 players, 500 rows), and every augmented row points back to a manual contract via `rank`.

In [ ]:
assert manual["origin"].eq("manual").all() and augmented["origin"].eq("span").all()
assert manual["player_id"].is_unique and len(manual) == 500
assert augmented["rank"].isin(manual["rank"]).all()
print("players:", manual["player_id"].nunique(),
      "| augmented rows per contract:", augmented.groupby("rank").size().describe()[["mean", "max"]].to_dict())
contracts.head()

## 2. Collection quality

Who collected what, and how many sources are links rather than named sites (a warning,
not an error, in our checks).

In [ ]:
is_url = contracts["source_url"].astype(str).str.startswith("http")
pd.DataFrame({"rows": contracts.groupby("collector").size(),
              "linked source": is_url.groupby(contracts["collector"]).mean().round(2)})

In [ ]:
contract_years = (pd.to_datetime(contracts["contract_end"]) - pd.to_datetime(contracts["contract_start"])).dt.days / 365.25
print(contract_years.describe().round(2).to_string())
print("flags: anchor_shifted", int(manual["anchor_shifted"].sum()), "| value_backfilled", int(manual["value_backfilled"].sum()))

## 3. The target and the survivor effect

`y_h1` is the log change in value over one season (0 = no change, 0.69 = doubled). These players
were picked because they are valuable **today**, so most of their past seasons were rises.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MANUAL, AUGMENTED = "#2a78d6", "#eb6834"
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(-1.5, 4, 45)
for d, c, lab in [(manual, MANUAL, "Manual"), (augmented, AUGMENTED, "Augmented")]:
    ax.hist(d["y_h1"].clip(-1.5, 4), bins=bins, histtype="step", lw=2, color=c, label=lab)
ax.axvline(0, color="grey", ls="--", lw=1)
ax.set(title="One-season log change in value", xlabel="log(value in 1 year / value now)", ylabel="rows")
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False); plt.show()
print(f"manual rows that rose: {(manual['y_h1'] > 0).mean():.0%}, median {manual['y_h1'].median():+.2f} log")

## 4. What moves with the target

In [ ]:
bands = pd.cut(manual["age"], [15, 19, 21, 23, 25, 27, 29])
by_age = manual.groupby(bands, observed=True)["y_h1"].agg(["median", "size"]).round(2)
by_pos = manual.groupby("position")["y_h1"].agg(["median", "size"]).round(2)
display(by_age, by_pos)

In [ ]:
num = manual.select_dtypes("number").drop(columns=["rank", "player_id", "season"])
corr = num.corrwith(num["y_h1"], method="spearman").drop("y_h1").dropna()
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(12)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index[::-1], top.values[::-1], color=MANUAL)
ax.axvline(0, color="grey", lw=1)
ax.set(title="Spearman correlation with the target, manual rows (top 12)", xlabel="rho")
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
print(corr[["age", "years_left", "contract_years", "value_change_12m"]].round(2).to_string())

**Current value dominates**, negatively: the cheaper a player in this sample, the more they
rose, which is the survivor effect again (cheap players here are future stars). Standing at the
club and playing time come next: a high `position_rank_at_club` number means a lower rank, and
those players had the most room to rise. The hand-collected contract features are weak on their
own. The ablation in the repo agrees: no feature group beyond age, position and value
measurably improves the model, because value and age already carry what these features carry.

## 5. Missing values

The 42 panel features are blank when the player was outside the 9 leagues we built the panel
for that season.

In [ ]:
miss = pd.concat([manual, augmented]).isna().mean().sort_values(ascending=False)
miss[miss > 0].round(3).head(15)

## 6. One player's value history (the input of the time-series model)

In [ ]:
pid = manual.sort_values("value_now", ascending=False).iloc[0]["player_id"]
h = history[history["player_id"] == pid]
row = manual[manual["player_id"] == pid].iloc[0]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.step(pd.to_datetime(h["date"]), h["market_value_in_eur"] / 1e6, where="post", color=MANUAL, lw=2)
ax.axvline(pd.to_datetime(row["anchor_date"]), color="grey", ls="--", lw=1)
ax.set(title=f"{row['player_name']}: Transfermarkt value (dashed: the sample's anchor)", ylabel="EUR million")
ax.spines[["top", "right"]].set_visible(False); plt.show()